# 🌸 Ejercicio: Clasificación de Iris con MLflow

## 🎯 Objetivos del Ejercicio

En este ejercicio práctico aprenderás a:

1. **Trabajar con un problema de clasificación** multiclase
2. **Aplicar MLflow** a un modelo de árbol de decisión
3. **Registrar experimentos** de forma profesional
4. **Evaluar y comparar** diferentes configuraciones
5. **Crear visualizaciones** para clasificación

---

## 🌺 El Dataset Iris

El **Iris Dataset** es uno de los datasets más famosos en Machine Learning, introducido por Ronald Fisher en 1936.

### 📊 Características del Dataset

| Aspecto | Descripción |
|---------|-------------|
| **Muestras** | 150 flores (50 por clase) |
| **Características** | 4 medidas físicas (cm) |
| **Clases** | 3 especies de iris |
| **Tipo** | Clasificación multiclase |
| **Dificultad** | ⭐⭐ (Principiante) |

### 🌸 Las 3 Especies de Iris

1. **Iris Setosa** (Clase 0)
2. **Iris Versicolor** (Clase 1)
3. **Iris Virginica** (Clase 2)

### 📏 Las 4 Características

1. **Sepal Length** (Longitud del sépalo)
2. **Sepal Width** (Ancho del sépalo)
3. **Petal Length** (Longitud del pétalo)
4. **Petal Width** (Ancho del pétalo)

---

## 🎯 Tu Misión

Construir un **Árbol de Decisión** que pueda clasificar correctamente las especies de iris basándose en sus medidas físicas, y documentar todo el proceso con MLflow.

---

## 🚀 ¡Empecemos!

In [0]:
import mlflow
import warnings
warnings.filterwarnings('ignore')


mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

# ⚠️ IMPORTANTE: Cambia esto por tu email de Databricks
email = 'gonzalovizoso97@gmail.com'  # Ejemplo: 'nombre.apellido@ejemplo.com'

# Validar que el email no esté vacío
if not email:
    print("⚠️  ADVERTENCIA: Debes configurar tu email antes de continuar")
    print("   Cambia la variable 'email' por tu email de Databricks")
else:
    # Configurar el experimento
    experiment_name = f"/Users/{email}/4-ejercicio-the-irish"
    mlflow.set_experiment(experiment_name)
    
    print("=" * 60)
    print("✅ MLflow configurado correctamente")
    print("=" * 60)
    print(f"📊 Experimento: {experiment_name}")
    print(f"🌸 Dataset: Iris (Clasificación)")
    print(f"🤖 Modelo: Decision Tree Classifier")
    print("=" * 60)

## ⚙️ Paso 1: Configuración de MLflow

Configuramos el experimento donde se registrarán todas las ejecuciones.

**🔴 IMPORTANTE**: Cambia el email vacío por tu email de Databricks.

## 📚 Paso 2: Importar Librerías

Importamos todas las herramientas necesarias para clasificación.

In [0]:
# Librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# MLflow para tracking
import mlflow
import mlflow.sklearn

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Todas las librerías importadas correctamente")
print("   - MLflow: Listo para tracking")
print("   - Scikit-learn: Modelos y métricas")
print("   - Matplotlib & Seaborn: Visualizaciones")

## 🌺 Paso 3: Cargar y Explorar el Dataset Iris

Cargaremos el famoso dataset de flores Iris y exploraremos sus características.

In [0]:
# TODO: Carga el dataset Iris y explora su estructura.
# Pistas:
# - Usa datasets.load_iris()
# - Crea un DataFrame con dataset.data y dataset.feature_names
# - Añade la columna target/especie
# - Muestra shape, clases, primeras filas y estadísticas descriptivas

dataset = datasets.load_iris()
df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
df['target'] = dataset.target
df['species'] = df['target'].map(dict(enumerate(dataset.target_names)))


print(df.shape)                    
print(dataset.target_names)        
display(df.head())
display(df.describe())
print(df['species'].value_counts())


### 📊 Visualización Exploratoria del Dataset

Visualicemos las relaciones entre las características para entender mejor los datos.

In [0]:
# TODO: Crea visualizaciones exploratorias del dataset Iris.
# Ideas:
# - Relación entre longitud/ancho del pétalo por especie
# - Distribución de características por especie
# - Boxplots por especie
# - Matriz de correlación

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

sns.scatterplot(data=df, x='petal length (cm)', y='petal width (cm)', hue='species', ax=axes[0, 0])
axes[0, 0].set_title('Pétalo: largo vs ancho')

sns.scatterplot(data=df, x='sepal length (cm)', y='sepal width (cm)', hue='species', ax=axes[0, 1])
axes[0, 1].set_title('Sépalo: largo vs ancho')

sns.boxplot(data=df, x='species', y='petal length (cm)', ax=axes[1, 0])
axes[1, 0].set_title('Largo del pétalo por especie')

sns.heatmap(df[dataset.feature_names].corr(), annot=True, cmap='coolwarm', ax=axes[1, 1])
axes[1, 1].set_title('Matriz de correlación')

plt.tight_layout()
plt.show()

# TODO: Escribe tus observaciones clave sobre separabilidad y correlaciones.

# SEPARABILIDAD
# - Setosa queda completamente aislada de las otras dos especies en el scatter del pétalo
#   (largo y ancho pequeños), así que es trivial de clasificar.
# - Versicolor y virginica están separadas en su mayor parte, pero se solapan en una zona
#   intermedia. Ahí es donde el modelo tendrá más probabilidad de equivocarse.
# - Las variables del pétalo separan mejor las clases que las del sépalo: en el scatter
#   del sépalo versicolor y virginica se mezclan bastante.
# - En el boxplot, el largo del pétalo de setosa no se solapa con las otras dos especies.

# CORRELACIONES
# - Largo y ancho del pétalo están muy correlacionados (~0.96): aportan información redundante.
# - El largo del sépalo también correlaciona fuerte con las variables del pétalo (~0.87 con el largo).
# - El ancho del sépalo es la variable más "independiente" y la que peor discrimina entre clases.

# CONCLUSIÓN
# - Es un problema casi linealmente separable, por lo que un árbol de decisión simple
#   debería funcionar muy bien, con los posibles errores concentrados entre versicolor y virginica.


## 🔀 Paso 4: Preparar los Datos

Dividimos el dataset en conjuntos de entrenamiento y prueba.

In [0]:
# TODO: Separa características (X) y etiquetas (y).
X = df[dataset.feature_names]
y = df['target']



# TODO: Divide el dataset en entrenamiento y prueba.
# Pistas:
# - Usa train_test_split
# - Reserva una parte para test
# - Usa random_state para reproducibilidad
# - Considera stratify=y para mantener la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

# TODO: Comprueba tamaños y distribución de clases en cada conjunto.


## 🌳 Paso 5: Entrenar Árbol de Decisión con MLflow

Ahora entrenaremos un modelo de **Decision Tree** y registraremos todo con MLflow.

### 🔑 Hiperparámetros del Árbol de Decisión

- **max_depth**: Profundidad máxima del árbol (evita overfitting)
- **max_features**: Número máximo de características a considerar por split
- Valores más altos = modelo más complejo = mayor riesgo de overfitting

In [0]:
# ========================================
# TODO: ENTRENAMIENTO CON MLFLOW
# ========================================

# TODO: Activa autologging de MLflow.
# mlflow.sklearn.autolog()

# TODO: Inicia un run con un nombre descriptivo.
# with mlflow.start_run(run_name="Decision Tree - Iris Classifier") as run:
#     TODO: Define hiperparámetros del árbol.
#     max_depth = ...
#     max_features = ...
#     random_state = ...
#
#     TODO: Crea y entrena DecisionTreeClassifier.
#     dt = DecisionTreeClassifier(...)
#     dt.fit(...)
#
#     TODO: Genera predicciones para train y test.
#     y_pred_train = ...
#     y_pred_test = ...
#
#     TODO: Calcula métricas de clasificación.
#     accuracy_test = ...
#     precision_test = ...
#     recall_test = ...
#     f1_test = ...
#
#     TODO: Evalúa con cross-validation.
#     cv_scores = ...
#
#     TODO: Registra métricas/parámetros relevantes en MLflow si no quedan registrados automáticamente.
#
#     TODO: Visualiza matriz de confusión y, opcionalmente, el árbol.
#
#     TODO: Escribe una breve interpretación de los resultados.

mlflow.sklearn.autolog()

with mlflow.start_run(run_name="Decision Tree - Iris Classifier") as run:
    # Hiperparámetros
    max_depth = 3
    max_features = 4
    random_state = 42

    # Modelo
    dt = DecisionTreeClassifier(
        max_depth=max_depth,
        max_features=max_features,
        random_state=random_state
    )
    dt.fit(X_train, y_train)

    # Predicciones
    y_pred_train = dt.predict(X_train)
    y_pred_test = dt.predict(X_test)

    # Métricas (macro = media de las 3 clases, necesario en multiclase)
    accuracy_train = accuracy_score(y_train, y_pred_train)
    accuracy_test = accuracy_score(y_test, y_pred_test)
    precision_test = precision_score(y_test, y_pred_test, average='macro')
    recall_test = recall_score(y_test, y_pred_test, average='macro')
    f1_test = f1_score(y_test, y_pred_test, average='macro')

    # Cross-validation solo sobre train
    cv_scores = cross_val_score(dt, X_train, y_train, cv=5)

    # Registro explícito con nombres claros
    mlflow.log_metric("test_accuracy", accuracy_test)
    mlflow.log_metric("test_precision_macro", precision_test)
    mlflow.log_metric("test_recall_macro", recall_test)
    mlflow.log_metric("test_f1_macro", f1_test)
    mlflow.log_metric("cv_accuracy_mean", cv_scores.mean())
    mlflow.log_metric("cv_accuracy_std", cv_scores.std())

    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred_test)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=dataset.target_names,
                yticklabels=dataset.target_names, ax=ax)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')
    ax.set_title('Matriz de confusión')
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.show()

    # Árbol (opcional)
    fig2, ax2 = plt.subplots(figsize=(14, 8))
    plot_tree(dt, feature_names=dataset.feature_names,
              class_names=dataset.target_names, filled=True, ax=ax2)
    mlflow.log_figure(fig2, "decision_tree.png")
    plt.show()

    print(classification_report(y_test, y_pred_test, target_names=dataset.target_names))
    print(f"Accuracy train: {accuracy_train:.3f} | test: {accuracy_test:.3f}")
    print(f"CV: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")


El árbol de decisión (max_depth=3) alcanza una accuracy de 0.967 en test y 0.933 ± 0.020 en validación cruzada, con precision, recall y F1 macro de 0.97. La diferencia entre train (0.983) y test es pequeña, por lo que no se aprecia overfitting. Setosa se clasifica perfectamente con una sola regla sobre el largo del pétalo; el único error ocurre entre versicolor y virginica, coherente con el solapamiento observado en el análisis exploratorio. El árbol solo utiliza variables del pétalo, lo que confirma que son las más discriminantes.

## 🎯 Reflexión del Ejercicio

Completa esta sección cuando termines el notebook.

### ✅ Comprueba que has trabajado

1. [ ] Exploración del dataset Iris
2. [ ] Separación train/test
3. [ ] Entrenamiento de un Árbol de Decisión
4. [ ] Evaluación con métricas de clasificación
5. [ ] Registro del experimento en MLflow
6. [ ] Interpretación de resultados

---

## 📚 Conceptos Clave - Clasificación

### 🎯 Métricas de Clasificación

| Métrica | Descripción | Cuándo Usarla |
|---------|-------------|---------------|
| **Accuracy** | % de predicciones correctas | Clases balanceadas |
| **Precision** | % de positivos correctos | Importante evitar falsos positivos |
| **Recall** | % de positivos encontrados | Importante encontrar todos los positivos |
| **F1-Score** | Media armónica de P y R | Balance entre precisión y recall |

### 🌳 Árbol de Decisión

**TODO: Explica con tus palabras:**
- ¿Qué ventajas tiene un árbol de decisión?
- ¿Qué riesgos tiene respecto a overfitting?
- ¿Cómo interpretarías una matriz de confusión multiclase?

---

## 🚀 Desafíos Adicionales

### 🎯 Desafío 1: Optimiza los Hiperparámetros

Prueba diferentes configuraciones y encuentra la mejor:

```python
# Experimenta con:
max_depth = [3, 5, 10, None]
max_features = [2, 3, 4, 'sqrt']
min_samples_split = [2, 5, 10]
```

**TODO:** ¿Qué combinación da el mejor balance entre accuracy y overfitting?

### 🎯 Desafío 2: Implementa Grid Search

Automatiza la búsqueda del mejor modelo con `GridSearchCV`.

### 🎯 Desafío 3: Compara con Otros Modelos

Entrena y compara Random Forest, SVM, KNN o Logistic Regression.

### 🎯 Desafío 4: Análisis de Errores

**TODO:** ¿Qué flores se confunden más y por qué?

### 🎯 Desafío 5: Reducción de Dimensionalidad

Usa PCA para reducir a 2 dimensiones y visualizar límites de decisión.


🌳 Árbol de Decisión
TODO: Explica con tus palabras:

Ventajas del árbol de decisión: es muy interpretable, porque el modelo es un conjunto de reglas if/else que se pueden leer y justificar (en nuestro caso, petal length <= 2.45 aísla setosa). No requiere escalar las variables, ya que cada división compara una sola variable con un umbral. Captura relaciones no lineales e interacciones entre variables mediante cortes sucesivos y es rápido de entrenar y de predecir.

Riesgo de overfitting: sin restricciones, el árbol crece hasta clasificar perfectamente el conjunto de entrenamiento, memorizando el ruido (alta varianza), y pequeños cambios en los datos pueden dar árboles muy distintos. Se controla con max_depth (limita los niveles), min_samples_split/min_samples_leaf, la poda por complejidad (ccp_alpha) o con ensembles como Random Forest. En nuestro caso, max_depth=3 da un accuracy de 0.983 en train y 0.967 en test, una diferencia pequeña que indica buena generalización. El parámetro max_features controla cuántas variables se evalúan en cada división (aquí 4, todas) y random_state fija la semilla para que los resultados sean reproducibles.

Matriz de confusión multiclase: las filas representan la clase real y las columnas la clase predicha. La diagonal contiene los aciertos y cada celda fuera de la diagonal indica qué clase real se confundió con cuál. En nuestro caso, setosa y virginica se clasifican sin errores y una muestra de versicolor se predice como virginica, coherente con el solapamiento entre esas dos especies.

In [0]:
# 🎨 DESAFÍOS OPCIONALES
# Usa este espacio para completar los desafíos del ejercicio.

# ========================================
# TODO: DESAFÍO 1 - Optimización manual
# ========================================
# configuraciones = [...]
# for config in configuraciones:
#     # Entrena, evalúa y registra cada configuración en MLflow
#     pass

# ========================================
# TODO: DESAFÍO 2 - Grid Search
# ========================================
# from sklearn.model_selection import GridSearchCV
# param_grid = {...}
# grid_search = GridSearchCV(...)
# grid_search.fit(...)

# ========================================
# TODO: DESAFÍO 3 - Comparación de modelos
# ========================================
# modelos = {...}
# resultados = []
# for nombre, modelo in modelos.items():
#     # Entrena y compara métricas
#     pass

# ========================================
# TODO: DESAFÍO 4 - Análisis de errores
# ========================================
# incorrect_indices = ...
# TODO: inspecciona las muestras mal clasificadas.

# ========================================
# TODO: DESAFÍO 5 - PCA y visualización 2D
# ========================================
# from sklearn.decomposition import PCA
# pca = PCA(n_components=2)
# X_pca = ...


In [0]:
# ========================================
# TODO: DESAFÍO 1 - Optimización manual
# ========================================
configuraciones = [
    # Referencia: árbol por defecto
    {"max_depth": None, "max_features": 4, "min_samples_split": 2},
    # Solo cambio max_depth
    {"max_depth": 3,  "max_features": 4, "min_samples_split": 2},
    {"max_depth": 5,  "max_features": 4, "min_samples_split": 2},
    {"max_depth": 10, "max_features": 4, "min_samples_split": 2},
    # Solo cambio max_features
    {"max_depth": None, "max_features": 2,      "min_samples_split": 2},
    {"max_depth": None, "max_features": 3,      "min_samples_split": 2},
    {"max_depth": None, "max_features": "sqrt", "min_samples_split": 2},
    # Solo cambio min_samples_split
    {"max_depth": None, "max_features": 4, "min_samples_split": 5},
    {"max_depth": None, "max_features": 4, "min_samples_split": 10},
]

resultados_manual = []
for i, config in enumerate(configuraciones, start=1):
    nombre = f"DT manual {i} | depth={config['max_depth']} feats={config['max_features']} split={config['min_samples_split']}"
    with mlflow.start_run(run_name=nombre):
        dt_cfg = DecisionTreeClassifier(**config, random_state=42)
        dt_cfg.fit(X_train, y_train)

        acc_train = accuracy_score(y_train, dt_cfg.predict(X_train))
        acc_test = accuracy_score(y_test, dt_cfg.predict(X_test))
        cv = cross_val_score(dt_cfg, X_train, y_train, cv=5)

        mlflow.log_metric("test_accuracy", acc_test)
        mlflow.log_metric("cv_accuracy_mean", cv.mean())
        mlflow.log_metric("overfit_gap", acc_train - acc_test)

        resultados_manual.append({
            **config,
            "profundidad_real": dt_cfg.get_depth(),
            "hojas": dt_cfg.get_n_leaves(),
            "train_acc": acc_train,
            "test_acc": acc_test,
            "cv_mean": cv.mean(),
            "gap": acc_train - acc_test,
        })

df_manual = pd.DataFrame(resultados_manual).sort_values("cv_mean", ascending=False)
display(df_manual)

In [0]:
df_manual = pd.DataFrame(resultados_manual).sort_values("cv_mean", ascending=False)

# Pasar a texto las columnas con tipos mezclados
for col in ["max_depth", "max_features"]:
    df_manual[col] = df_manual[col].astype(str)

display(df_manual)



¿Qué combinación da el mejor balance entre accuracy y overfitting?

Se probó cada hiperparámetro por separado, partiendo del árbol por defecto (max_depth=None, max_features=4, min_samples_split=2), que llega a train = 1.0 con un gap de 0.067. Las configuraciones con max_features=2 o 'sqrt' (equivalentes, porque √4 = 2) obtuvieron la mejor CV (0.958), pero generan los árboles más grandes (profundidad 7, 10 hojas), memorizan el train (1.0) y tienen el peor gap (0.067).

El mejor balance lo dan max_depth=3 y min_samples_split=5 o 10, que producen el mismo árbol (profundidad 3, 5 hojas): train 0.983, test 0.967 y un gap de solo 0.017. Su CV (0.933) es inferior a la de max_features=2, pero esa diferencia equivale a unas 3 muestras de 120 y queda dentro del ruido (desviación de la CV ≈ 0.02), por lo que no es concluyente. Ante un empate estadístico se elige el modelo más simple y con menos overfitting. Los valores max_depth=5 y 10 dan el mismo resultado que None: el árbol se detiene solo en profundidad 5, así que esos límites nunca llegan a actuar.

Nota: al variar un hiperparámetro cada vez, no se han evaluado combinaciones; eso se aborda en el Desafío 2 con Grid Search.

In [0]:
# ========================================
# TODO: DESAFÍO 2 - Grid Search
# ========================================

from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [3, 5, 10, None],
    "max_features": [2, 3, 4, "sqrt"],
    "min_samples_split": [2, 5, 10],
}

with mlflow.start_run(run_name="GridSearch - Decision Tree"):
    grid_search = GridSearchCV(
        DecisionTreeClassifier(random_state=42),
        param_grid,
        cv=5,
        scoring="accuracy",
        return_train_score=True,
    )
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    test_acc = accuracy_score(y_test, best_model.predict(X_test))

    mlflow.log_metric("best_cv_accuracy", grid_search.best_score_)
    mlflow.log_metric("test_accuracy_best_model", test_acc)

print("Mejores parámetros:", grid_search.best_params_)
print(f"CV: {grid_search.best_score_:.3f} | Test: {test_acc:.3f}")

# Top 10 combinaciones (con texto en las columnas de tipos mezclados, para que display() no falle)
cv_res = pd.DataFrame(grid_search.cv_results_)
cols = ["param_max_depth", "param_max_features", "param_min_samples_split",
        "mean_test_score", "std_test_score", "mean_train_score"]
top = cv_res.sort_values("rank_test_score").head(10)[cols].copy()
for c in ["param_max_depth", "param_max_features", "param_min_samples_split"]:
    top[c] = top[c].astype(str)
top.columns = ["max_depth", "max_features", "min_samples_split", "cv_mean", "cv_std", "train_mean"]
display(top.round(3))

Resultado del Grid Search

Se evaluaron las 48 combinaciones de hiperparámetros (4 valores de max_depth × 4 de max_features × 3 de min_samples_split) con validación cruzada de 5 folds. Las mejores combinaciones obtienen una CV de 0.967 (± 0.031) con max_depth=3 y max_features=2 (o 'sqrt', equivalente aquí porque √4 = 2). El valor de min_samples_split no influye: con esa profundidad el umbral nunca impide una división, por lo que las 6 primeras filas corresponden en la práctica al mismo modelo.

Además, en esas combinaciones el accuracy de train (0.967) coincide con el de validación, lo que indica que no hay overfitting. Las configuraciones sin límite de profundidad (None) o con max_depth=10 alcanzan 0.958 de CV, y en algunas el train llega a 1.0, es decir, memorizan los datos de entrenamiento. La diferencia entre 0.967 y 0.958 equivale a una única muestra de las 120 de entrenamiento, por lo que está dentro del ruido; a igualdad estadística se prefiere el modelo más simple.

Mejor combinación: max_depth=3, max_features=2 (o 'sqrt') y min_samples_split=2 (valor por defecto). El mejor modelo ([best_params_]) obtiene un accuracy de [test] en el conjunto de test. Frente al árbol inicial (max_depth=3, max_features=4, CV = 0.933), restringir max_features a 2 mejora la CV en unos 3 puntos. Con un dataset de 150 muestras conviene tomar esa mejora con cautela, porque diferencias de un par de muestras pueden deberse al azar.

In [0]:
# ========================================
# TODO: DESAFÍO 3 - Comparación de modelos
# ========================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

modelos = {
    "Decision Tree": DecisionTreeClassifier(max_depth=3, max_features=2, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": make_pipeline(StandardScaler(), SVC(random_state=42)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=200, random_state=42)),
}

resultados = []
for nombre, modelo in modelos.items():
    with mlflow.start_run(run_name=f"Compare - {nombre}"):
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        cv = cross_val_score(modelo, X_train, y_train, cv=5)

        fila = {
            "modelo": nombre,
            "train_acc": round(accuracy_score(y_train, modelo.predict(X_train)), 3),
            "test_acc": round(accuracy_score(y_test, y_pred), 3),
            "test_f1_macro": round(f1_score(y_test, y_pred, average="macro"), 3),
            "cv_mean": round(cv.mean(), 3),
            "cv_std": round(cv.std(), 3),
        }
        mlflow.log_param("model_type", nombre)
        mlflow.log_metrics({k: v for k, v in fila.items() if k != "modelo"})
        resultados.append(fila)

df_modelos = pd.DataFrame(resultados).sort_values("cv_mean", ascending=False)
display(df_modelos)

Comparación de modelos

Se entrenaron cinco modelos con el mismo conjunto de entrenamiento y test y la misma validación cruzada de 5 folds: árbol de decisión (max_depth=3, max_features=2), Random Forest (100 árboles), SVM, KNN (k=5) y regresión logística. En los tres últimos las variables se estandarizaron dentro de un pipeline, porque dependen de distancias o magnitudes, y así se evita fuga de datos.

El árbol, la SVM y KNN empatan en validación cruzada (0.967), la regresión logística queda en 0.958 y el Random Forest en 0.950. En test, la SVM logra 0.967 (1 error de 30), el árbol, KNN y la regresión logística 0.933 (2 errores) y el Random Forest 0.900 (3 errores). Con solo 30 muestras de test, cada error supone 3,3 puntos, por lo que estas diferencias no son estadísticamente concluyentes: en un problema tan sencillo como Iris todos los modelos rinden de forma parecida. El Random Forest alcanza train = 1.0, algo habitual en este modelo, pero su rendimiento en validación (0.950) no mejora al de un árbol simple, aunque es el más estable entre folds (desviación 0.017).

La SVM es marginalmente la mejor (CV 0.967, test 0.967 y poca diferencia con train). Sin embargo, dado que las diferencias entre modelos quedan dentro del ruido, el árbol de decisión resulta igual de competitivo y ofrece la ventaja de la interpretabilidad, pues sus reglas pueden leerse y justificarse directamente. Para un problema de este tipo no compensa un modelo más complejo como el Random Forest.

In [0]:
mlflow.sklearn.autolog(disable=True)   # evita que el fit de aquí cree un run automático

modelo_final = DecisionTreeClassifier(max_depth=3, max_features=4, random_state=42)
modelo_final.fit(X_train, y_train)
y_pred = modelo_final.predict(X_test)

# 1. Muestras mal clasificadas
incorrect_indices = np.where(y_pred != y_test.values)[0]
nombres = dict(enumerate(dataset.target_names))

errores = X_test.iloc[incorrect_indices].copy()
errores["real"] = y_test.iloc[incorrect_indices].map(nombres)
errores["predicha"] = pd.Series(y_pred[incorrect_indices], index=errores.index).map(nombres)
print(f"Errores: {len(errores)} de {len(y_test)}")
display(errores)

# 2. Perfil medio de cada especie, para comparar
display(df.groupby("species")[list(dataset.feature_names)].mean().round(2))

# 3. Dónde caen los errores en el espacio del pétalo
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=df, x="petal length (cm)", y="petal width (cm)", hue="species", alpha=0.6, ax=ax)
ax.scatter(errores["petal length (cm)"], errores["petal width (cm)"],
           s=250, facecolors="none", edgecolors="red", linewidths=2, label="error")
ax.legend()
plt.show()

Análisis de errores

El modelo comete un único error en las 30 muestras de test: una flor versicolor (sépalo 6.7 × 3.0 cm, pétalo 5.0 × 1.7 cm) clasificada como virginica. Setosa nunca se confunde con las otras especies. Las flores que más se confunden son versicolor y virginica, porque sus medidas se solapan. Esta muestra tiene un pétalo mayor de lo habitual en versicolor (largo 5.0 frente a una media de 4.26 y ancho 1.7 frente a 1.33) y se acerca más al perfil medio de virginica. En el gráfico cae justo en la zona donde se mezclan los puntos de ambas especies.

El árbol clasifica mediante cortes sucesivos: esta flor supera por poco margen los umbrales de ancho de pétalo (1.7 frente a 1.65) y de largo de pétalo (5.0 frente a 4.85), por lo que acaba en la hoja de virginica. Al haber un solo error no se puede afirmar que la confusión ocurra siempre en esa dirección, pero el resultado es coherente con el solapamiento observado en el análisis exploratorio.

In [0]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Estandarizar y proyectar a 2 componentes (ajustando solo con train)
scaler = StandardScaler().fit(X_train)
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(scaler.transform(X_train))
X_test_pca = pca.transform(scaler.transform(X_test))

var_ratio = pca.explained_variance_ratio_
print("Varianza explicada por componente:", var_ratio.round(3), "| total:", round(var_ratio.sum(), 3))

# Peso de cada variable original en cada componente
loadings = pd.DataFrame(pca.components_.T, index=dataset.feature_names, columns=["PC1", "PC2"]).round(2)
display(loadings.reset_index().rename(columns={"index": "variable"}))

# 2. Árbol sobre 2 componentes vs árbol sobre las 4 variables originales
dt_pca = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train_pca, y_train)
dt_4d = DecisionTreeClassifier(max_depth=3, max_features=4, random_state=42).fit(X_train, y_train)
acc_pca = accuracy_score(y_test, dt_pca.predict(X_test_pca))
acc_4d = accuracy_score(y_test, dt_4d.predict(X_test))
print(f"Accuracy test | 4 variables: {acc_4d:.3f} | 2 componentes PCA: {acc_pca:.3f}")

# 3. Malla para pintar las regiones de decisión
x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = dt_pca.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

colores = sns.color_palette("Set2")[:3]
with mlflow.start_run(run_name="PCA 2D - Decision Tree"):
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5, 2.5], colors=colores, alpha=0.3)
    for i, nombre in enumerate(dataset.target_names):
        m_tr = y_train.values == i
        m_te = y_test.values == i
        ax.scatter(X_train_pca[m_tr, 0], X_train_pca[m_tr, 1], color=colores[i],
                   edgecolor="k", label=f"{nombre} (train)")
        ax.scatter(X_test_pca[m_te, 0], X_test_pca[m_te, 1], color=colores[i],
                   edgecolor="k", marker="^", s=90, label=f"{nombre} (test)")
    ax.set_xlabel(f"PC1 ({var_ratio[0]:.0%} de la varianza)")
    ax.set_ylabel(f"PC2 ({var_ratio[1]:.0%} de la varianza)")
    ax.set_title("Límites de decisión del árbol sobre 2 componentes PCA")
    ax.legend(fontsize=8)

    mlflow.log_param("n_components", 2)
    mlflow.log_metric("explained_variance_total", var_ratio.sum())
    mlflow.log_metric("test_accuracy_pca", acc_pca)
    mlflow.log_metric("test_accuracy_4_variables", acc_4d)
    mlflow.log_figure(fig, "pca_decision_boundary.png")
    plt.show()

PCA y límites de decisión en 2D

Las dos primeras componentes principales retienen el 96 % de la varianza (PC1 73 % y PC2 23 %), así que la proyección a 2D conserva casi toda la información del dataset. En el gráfico, setosa queda claramente aislada en el lado izquierdo de PC1, mientras que versicolor y virginica se tocan en la zona central-derecha.

Los límites de decisión del árbol son tres bandas verticales: solo usa PC1 para separar las tres clases y no necesita PC2. Es la firma de un árbol de decisión, que solo hace cortes paralelos a los ejes (nunca diagonales ni curvas). Con 2 componentes la accuracy en test es 0.900 (3 errores en 30), frente a 0.967 (1 error) con las 4 variables originales. Como setosa está aislada, los fallos solo pueden ocurrir en la frontera entre versicolor y virginica, donde las nubes de puntos se solapan.

Comprimir a 2D supone perder algo de información y de interpretabilidad, porque PC1 y PC2 son combinaciones de las variables originales sin unidades físicas. A cambio, permite visualizar todo el problema en un plano. La diferencia entre ambos modelos son 2 muestras de 30, por lo que conviene interpretarla con cautela.